# DSA Regelwiki Crawler

Dieses Notebook crawlt das DSA Ulisses Regelwiki und speichert jeden Artikel als JSON in `data/raw/`.

**Ablauf:**
1. Sitemap laden → Liste aller URLs
2. Hilfsfunktionen definieren (Breadcrumb, Artikeltext)
3. Alle URLs crawlen und als JSON speichern
4. Data Cleaning: leere Dateien und Suchseiten entfernen

## 1. Imports

In [1]:
import requests                    # HTTP-Requests: Webseiten und XML herunterladen
from bs4 import BeautifulSoup      # HTML/XML parsen und durchsuchen
import time                        # Wartezeit zwischen Requests (Server schonen)
import json                        # JSON lesen und schreiben
from pathlib import Path           # Plattformunabhängige Dateipfade

## 2. Sitemap laden

Die Sitemap ist eine XML-Datei die alle URLs der Website auflistet.
Wir parsen sie mit BeautifulSoup und extrahieren alle `<loc>`-Tags (= URLs).

In [2]:
SITEMAP_URL = "https://dsa.ulisses-regelwiki.de/sitemap.xml"

# GET-Request: Sitemap herunterladen
# response.text enthält den rohen XML-Inhalt als String
response = requests.get(SITEMAP_URL)

# BeautifulSoup parst das XML in einen durchsuchbaren Baum
# 'xml' als Parser weil es sich um XML handelt (nicht HTML)
soup = BeautifulSoup(response.text, "xml")

# Alle <loc>-Tags extrahieren → das sind die Artikel-URLs
all_urls = [loc.text for loc in soup.find_all("loc")]

# Nur HTML-Seiten crawlen (keine Homepage, keine anderen Formate)
crawl_urls = [u for u in all_urls if u.endswith(".html")]

print(f"Gesamt URLs in Sitemap: {len(all_urls)}")
print(f"Zu crawlen (nur .html): {len(crawl_urls)}")
print("\nBeispiel-URLs:")
print(*crawl_urls[:5], sep="\n")

Gesamt URLs in Sitemap: 5936
Zu crawlen (nur .html): 5935

Beispiel-URLs:
https://dsa.ulisses-regelwiki.de/regeln.html
https://dsa.ulisses-regelwiki.de/alternative_regeln.html
https://dsa.ulisses-regelwiki.de/AlternativRegel_1W20-Probe.html
https://dsa.ulisses-regelwiki.de/AlternativRegel_1W20-Probe_Methode3.html
https://dsa.ulisses-regelwiki.de/AlternativRegel_1W20-Probe_Methode3s.html


## 3. Hilfsfunktionen

### 3a. Breadcrumb extrahieren

Jede Seite hat oben einen Navigationspfad (Breadcrumb), z.B.:

`DSA Regel-Wiki › Rüstkammer › Waffen › Schwerter › Reguläre Schwerter`

Daraus gewinnen wir die Kategorien des Artikels.
Die Root-Ebene (`DSA Regel-Wiki`) und der Artikel selbst (CSS-Klasse `active`) werden übersprungen.

In [3]:
def extract_breadcrumb(soup):
    """Extrahiert den Kategoriepfad aus dem Breadcrumb-Nav der Seite.
    
    Gibt eine Liste zurück, z.B.:
    ['Rüstkammer', 'Waffen', 'Schwerter', 'Reguläre Schwerter']
    
    Leere Liste wenn kein Breadcrumb vorhanden (z.B. bei Toplevel-Seiten).
    """
    breadcrumb = soup.find("div", class_="mod_breadcrumb")
    if not breadcrumb:
        return []
    
    categories = []
    for li in breadcrumb.find_all("li"):
        # Trennzeichen (›) haben CSS-Klasse 'sep' → überspringen
        if "sep" in li.get("class", []):
            continue
        # Aktueller Artikel hat CSS-Klasse 'active' → überspringen
        if "active" in li.get("class", []):
            continue
        # Root-Ebene 'DSA Regel-Wiki' überspringen
        a = li.find("a")
        if a and a.get("title") != "DSA Regel-Wiki":
            categories.append(a.get("title"))
    
    return categories

### 3b. Einzelne Seite scrapen

Pro URL:
- Titel aus `<title>`-Tag extrahieren (Suffix " - DSA Regel-Wiki" abschneiden)
- Kategorien via Breadcrumb extrahieren
- Artikeltext aus `<div class="mod_article">` extrahieren
  - Erste Zeile ist immer der Titel (Duplikat) → wird entfernt

Rückgabe: Dictionary mit `url`, `title`, `categories`, `text`

In [4]:
def scrape_page(url):
    """Lädt eine Wiki-Seite und extrahiert Titel, Kategorien und Text.
    
    Rückgabe: {'url': ..., 'title': ..., 'categories': [...], 'text': ...}
    """
    r = requests.get(url)
    # 'html.parser' für HTML-Seiten (im Gegensatz zu 'xml' für die Sitemap)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Titel: aus <title>-Tag, Wiki-Suffix entfernen
    title = soup.find("title").text.strip().replace(" - DSA Regel-Wiki", "")
    
    # Kategorien aus Breadcrumb
    categories = extract_breadcrumb(soup)
    
    # Artikeltext: mod_article enthält den eigentlichen Seiteninhalt
    # get_text() extrahiert allen Text ohne HTML-Tags
    article = soup.find("div", class_="mod_article")
    if article:
        lines = article.get_text(separator="\n", strip=True).split("\n")
        # Erste Zeile ist immer der Titel (Duplikat aus H1) → entfernen
        text = "\n".join(lines[1:]) if lines[0] == title else "\n".join(lines)
    else:
        text = ""
    
    return {"url": url, "title": title, "categories": categories, "text": text}

## 4. Crawler ausführen

Iteriert über alle URLs, speichert jeden Artikel als JSON in `data/raw/`.

- Dateiname wird aus der URL abgeleitet (Slashes → Underscores, damit keine Unterordner entstehen)
- `time.sleep(0.5)`: 0.5 Sekunden Pause zwischen Requests um den Server nicht zu überlasten
- Fehler werden gesammelt und am Ende ausgegeben
- Fortschritt alle 100 Artikel

**Laufzeit: ~50 Minuten für ~5900 URLs**

In [5]:
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)  # Ordner anlegen falls nicht vorhanden

errors = []

for i, url in enumerate(crawl_urls):
    try:
        result = scrape_page(url)
        
        # Dateiname aus URL ableiten:
        # Basis-URL entfernen, Slashes durch Underscores ersetzen
        # z.B. waffen/raufen/achaz.html → waffen_raufen_achaz.json
        filename = (
            url.replace("https://dsa.ulisses-regelwiki.de/", "")
               .replace("/", "_")
               .replace(".html", ".json")
        )
        filepath = output_dir / filename
        
        # Als JSON speichern (ensure_ascii=False: Umlaute direkt, nicht escaped)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
        if i % 100 == 0:
            print(f"{i}/{len(crawl_urls)} – {result['title']}")
        
        time.sleep(0.5)  # Server schonen
    
    except Exception as e:
        errors.append({"url": url, "error": str(e)})
        print(f"FEHLER bei {url}: {e}")

print(f"\nFertig. Fehler: {len(errors)}")

0/5935 – Regeln
100/5935 – Ergebnis
200/5935 – QS 2
300/5935 – Herstellungstabelle (Fesseln)
400/5935 – Runenhandwerker / Bannglyphenhandwerker
500/5935 – Masse und Gewicht von Dämonen
600/5935 – Alrik Dagabor
700/5935 – Waldwildnis (Stufe II)
800/5935 – Geld verdienen
900/5935 – Entweihen
1000/5935 – Kampfsonderfertigkeiten
1100/5935 – Strukturschaden
1200/5935 – Kritische Treffer und Kritische Fehlschläge
1300/5935 – Achaz-Rha
1400/5935 – Firungeweihter
1500/5935 – Havener Krieger
1600/5935 – Graumagier (Halle der Antimagie zu Kuslik)
1700/5935 – Glückskind
1800/5935 – Licht des Madamals
1900/5935 – Suche
2000/5935 – Dämonische Fertigkeiten
2100/5935 – Zauberei mit Lebenskraft
2200/5935 – Faszinierende Kleidung
2300/5935 – Volumenerweiterung des Rings I-V
2400/5935 – Zauberspeicher des Erzmagus
2500/5935 – Hofzeremoniell
2600/5935 – Befehl Überrennen
2700/5935 – Kopist
2800/5935 – Scharfschütze I-II
2900/5935 – Hardas-Stil
3000/5935 – Weg d. Okkultisten
3100/5935 – Furchtbote
3200/59

## 5. Data Cleaning

Nach dem Crawlen gibt es zwei Kategorien unbrauchbarer Dateien:
- **Leere Dateien**: Seiten ohne Artikeltext (Weiterleitungen, Fehlerseiten)
- **Suchseiten**: Wiki-Suchformular, kein inhaltlicher Wert

Kurze Einträge (<100 Zeichen) werden **behalten** – z.B. "Kugeln" oder "Holzschale" sind valide Kurzeinträge.

In [13]:
# Überblick: Wie viele Dateien sind leer, kurz oder normal?
files = list(output_dir.glob("*.json"))
print(f"Dateien gesamt: {len(files)}")

empty, short, normal = 0, 0, 0

for f in files:
    with open(f, encoding="utf-8") as fh:
        data = json.load(fh)
    length = len(data["text"])
    if length == 0:
        empty += 1
    elif length < 100:
        short += 1
    else:
        normal += 1

print(f"Leer (0 Zeichen):         {empty}")
print(f"Sehr kurz (<100 Zeichen): {short}")
print(f"Normal (>=100 Zeichen):   {normal}")

Dateien gesamt: 5149
Leer (0 Zeichen):         0
Sehr kurz (<100 Zeichen): 10
Normal (>=100 Zeichen):   5139


In [10]:
old_files = []
new_files = []

for f in files:
    with open(f, encoding="utf-8") as fh:
        data = json.load(fh)
    if "categories" in data:
        new_files.append(f)
    else:
        old_files.append(f)

print(f"Alt (ohne categories): {len(old_files)}")
print(f"Neu (mit categories):  {len(new_files)}")

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\1-wurf-anwendung-w6.json'

In [9]:
for f in old_files:
    f.unlink()

print(f"Entfernt: {len(old_files)}")
print(f"Verbleibend: {len(list(output_dir.glob('*.json')))}")

Entfernt: 1102
Verbleibend: 5931


In [12]:
# Leere Dateien und Suchseiten löschen
kept, removed = 0, 0

for f in files:
    with open(f, encoding="utf-8") as fh:
        data = json.load(fh)
    
    if len(data["text"]) == 0 or data["title"] == "Suche":
        f.unlink()   # Datei von der Festplatte löschen
        removed += 1
    else:
        kept += 1

print(f"Entfernt: {removed}")
print(f"Behalten: {kept}")

Entfernt: 782
Behalten: 5149
